# 01 - Exploratory Data Analysis & Injected Production Drift

Inspects the reference credit-risk dataset (`10,000` rows) and six monthly production batches (`2026-01` to `2026-06`, `2,000` rows each) to examine feature distributions, missingness growth, and out-of-range anomalies.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.data_generator import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    generate_reference_and_production_batches,
)

ref_df, prod_df = generate_reference_and_production_batches(seed=42)
print(f"Reference shape: {ref_df.shape}, Production shape: {prod_df.shape}")
ref_df.head()

In [ ]:
missing_by_batch = (
    prod_df.groupby("batch_id")[list(NUMERIC_FEATURES) + list(CATEGORICAL_FEATURES)]
    .apply(lambda g: g.isna().mean().mean())
    .rename("missing_rate")
)
default_rate_by_batch = prod_df.groupby("batch_id")["default"].mean().rename("default_rate")
pd.concat([missing_by_batch, default_rate_by_batch], axis=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(data=ref_df, x="credit_score", label="reference", ax=axes[0])
sns.kdeplot(data=prod_df[prod_df["batch_id"] == "2026-06"], x="credit_score", label="2026-06", ax=axes[0])
axes[0].set_title("Credit Score Shift: Reference vs 2026-06")
axes[0].legend()

sns.kdeplot(data=ref_df, x="debt_to_income", label="reference", ax=axes[1])
sns.kdeplot(data=prod_df[prod_df["batch_id"] == "2026-06"], x="debt_to_income", label="2026-06", ax=axes[1])
axes[1].set_title("Debt-to-Income Shift: Reference vs 2026-06")
axes[1].legend()
plt.tight_layout()